### Goal of this notebook
- How to stream the dataset in batch size and hive
- Sample the data based on the techniques
- The augmentation strategies
- Tokenisation


In [1]:
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.compute as pc
import ray
import os


from dataclasses import dataclass
from datasets import load_dataset, Audio, interleave_datasets
from itertools import product
from pyarrow import fs
from pydantic import BaseModel, Field
from ray.data.datasource.partitioning import Partitioning

In [2]:
DATASET_REPO = "nolimitsxl/wazobia-asr-data"
DATASET_DIR = "data"
METRICS_PATH = "metrics/metrics.tsv"
URI = "hf://datasets/nolimitsxl/wazobia-asr-data/data"

### Random Generator
Pick one seeded generator (e.g. numpy.random.Generator) used everywhere randomness is needed (mixer draws, buffer shuffling). Verify you can get/set its state and that restoring it reproduces the exact same next-N draws.

In [3]:
rng = np.random.default_rng(seed=42)
files = np.arange(1, 14)
rng.shuffle(files)
print(files)
rng = np.random.default_rng(42)
files = np.arange(1, 14)
rng.shuffle(files)
files

[ 1 13  8 11 10  4  7  6  3  5 12  2  9]


array([ 1, 13,  8, 11, 10,  4,  7,  6,  3,  5, 12,  2,  9])

In [4]:
rng = np.random.default_rng(seed=42)
files = np.arange(1, 14)
rng.shuffle(files)
print(files)

state = rng.bit_generator.state

rng2 = np.random.default_rng()
rng2.bit_generator.state = state

files2 = np.arange(1,14)
rng2.shuffle(files2)

print(files2)

[ 1 13  8 11 10  4  7  6  3  5 12  2  9]
[ 8 13  5  3 11  6  2  9 10  7 12  4  1]


### Partition discovery.
Enumerate fragments via dataset.get_fragments(), extract (corpus, language) from each fragment's partition keys. Build {(corpus, language): [fragments]}. Verify: every expected leaf appears, no fragment double-counted.

In [5]:
dataset = ds.dataset(URI, format="parquet", partitioning="hive")

In [6]:
def get_partition_query(schema,comb):
    filter = None
    for key, value in zip(schema, comb):
        q = pc.field(key) == value
        if filter is None:
            filter = q
            continue
        filter = filter & q 
    return filter

dictionaries = [item.to_pylist() for item in dataset.partitioning.dictionaries]
products = product(*dictionaries)
fragments_enum = {}

for comb in products:
    q = get_partition_query(dataset.partitioning.schema.names, comb)
    fragments = list(dataset.get_fragments(q))
    if len(fragments) == 0:
        continue
    fragments = [f.path for f in fragments]
    fragments_enum[comb] = fragments


In [7]:
fragments_enum_2 = {
}
data_config = {}
partition_cols = dataset.partitioning.schema.names
for frag in dataset.get_fragments():
    keys = ds.get_partition_keys(frag.partition_expression)
    key = (keys[schema_name] for schema_name in partition_cols)
    if keys["split"] not in fragments_enum_2:
        fragments_enum_2.setdefault(keys["split"], {})
    fragments_enum_2[keys["split"]].setdefault(tuple(key),[]).append(frag.path)

### Sampler

In [8]:
def smooth(hours, beta):
    w = hours[hours > 0] ** beta
    return w / w.sum()

def leaf_weights(df, beta_corpus, beta_language, split="train"):
    df = df[df["split"] == split]
    p_corpus = smooth(df.groupby("corpus")["hours"].sum(), beta_corpus)
    out = {}
    for corpus, p_c in p_corpus.items():
        sub = df[df["corpus"] == corpus].groupby("language")["hours"].sum()
        for lang, p_l in smooth(sub, beta_language).items():
            out[(corpus, split,lang)] = p_c * p_l
    return out

### Per-leaf row iterator.
 For one leaf, flatten to_batches() into a row-at-a-time generator. Track (fragment_index, row_offset) internally. Verify: total rows read matches the leaf's known row count.

Stream Data for each path, per split interleave the dataset based on the sampling probability, parse the dataset to the dataloader.
- Each split data: interleaved based on the corpus & language
train: corpus_lang_1 (Dataset), corpus_lang_2(Dataset)
- dict(split, list[dict((corpora, lang), Loaded Data)])

In [9]:
next(iter(fragments_enum_2["train"]))

('Igbo_sync', 'train', 'ibo')

In [10]:
hf, path = fs.FileSystem.from_uri("hf://datasets/nolimitsxl")
DATASET_REPO = "nolimitsxl/wazobia-asr-data"
DATASET_DIR = "data"
COLS_TO_SEL = ["audio", "text", "duration_sec"]
METRICS_PATH = "../data/metrics/metrics.tsv"

def load_fragments(keys, values):
    dir = "/".join([f"{k}={v.replace(" ", "%20")}" for k,v in zip(keys, values)])
    dir = DATASET_DIR + "/" + dir
    data = load_dataset(DATASET_REPO, data_dir=dir, streaming=True)
    data = data.select_columns(COLS_TO_SEL)
    data = data.cast_column("audio", Audio(sampling_rate=16_000))
    return data["train"]




In [11]:
metrics_tsv = pd.read_csv(METRICS_PATH, delimiter="\t")
leaf_distr =leaf_weights(metrics_tsv, 0.25, 0.2)

In [12]:
weights_seq = []
data_seq = []
for leaf in leaf_distr:
    data = load_fragments(partition_cols, leaf)
    data_seq.append(data)
    weights_seq.append(leaf_distr[leaf])

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/223 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/217 [00:00<?, ?it/s]

In [13]:
num_threads = num_threads = min(32, (os.cpu_count() or 1) + 4)
num_threads

16

In [20]:
train_data = interleave_datasets(data_seq[:3], probabilities=[0.33,0.33,0.34], seed=42, stopping_strategy="all_exhausted")
train_data = train_data.decode(num_threads=num_threads)

In [ ]:
x = next(iter(train_data))

'peer closed connection without sending complete message body (received 9340292 bytes, expected 85757942)' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DIgbo_sync/split%3Dtrain/language%3Dibo/91d3b03c2015612422dc-00000.parquet
Retrying in 8s [Retry 4/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DIgbo_sync/split%3Dtrain/language%3Dibo/91d3b03c2015612422dc-00000.parquet
Retrying in 8s [Retry 5/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DOpen%2520SLR/split%3Dtrain/language%3Deng/78bc25f94885203c26ef-00000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://hu

In [ ]:
for _ in x:
    print(_)

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DOpen%2520SLR/split%3Dtrain/language%3Deng/78bc25f94885203c26ef-00000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DIgbo_sync/split%3Dtrain/language%3Dibo/91d3b03c2015612422dc-00000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DOpen%2520SLR/split%3Dtrain/language%3Deng/78bc25f94885203c26ef-00000.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a

RuntimeError: This event loop is already running

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/nolimitsxl/wazobia-asr-data/resolve/6df77bf0a44110a05adc42ada0d7fc925375b02a/data/corpus%3DIgbo_sync/split%3Dtrain/language%3Dibo/91d3b03c2015612422dc-00000.parquet
Retrying in 8s [Retry 4/5].


### Augmentation


#### Noise Injection